# Predoctoral Research Assistant Assessment - Khushi Desai
### Purpose: Merge ACS and CDC PLACES data at census tract level for analysis
This script documents all data cleaning decisions:
1. ACS data acquisition and processing
2. CDC PLACES 2021 release loading (it was downloaded directly from the website)
3. Merging on census tract identifier
4. Handling missing data
5. Creating analysis-ready dataset

In [30]:
#load necessary libraries
import os
import requests
import pandas as pd
import json

### Helper code chunk
Uncomment (remove the hashtag) the next code chunk if you don't have pandas, requests or numpy installed in your environment.
Note: you may need to restart the kernel to use updated packages.

In [ ]:
# pip install pandas requests numpy 

## Step 1 - American Community Survey (ACS) 5-year estimates, 2015–2019 data acquisition using census API
Fetch your own API key from the official census website - https://api.census.gov/data/key_signup.html 

Source: U.S. Census Bureau
Geographic level: Census tract
Time period: 2015-2019 (5-year rolling average)

To reproduce: Get your own Census API key from https://api.census.gov/data/key_signup.html

In [ ]:

api_key = os.environ.get('YOUR_API_KEY')

In [33]:
# Define variables following CDC Social Vulnerability Index (SVI) Theme 1: Socioeconomic Status
# Theorectical approach documented on: https://www.arcgis.com/home/item.html?id=ed69e728e4bf49da9a40cd50d1e45ff7
acs_variables = {
    'B17001_002E': 'Population_Below_Poverty', 'B17001_001E': 'Total_Population_Poverty', 'B19013_001E': 'Median_Household_Income', #economic hardship
    'B06009_001E': 'Population_25Plus', 'B06009_002E': 'Population_Less_9th_Grade', #education
    'B23025_001E': 'Civilian_Labor_Force', 'B23025_005E': 'Unemployed', 'B23025_003E': 'Employed', #employment
    'B25002_001E': 'Total_Housing_Units', 'B25002_003E': 'Vacant_Housing_Units', #housing stability
}

In [ ]:
var_list = ','.join(acs_variables.keys())
 
#census API endpoint for ACS 5-year
api_url = "https://api.census.gov/data/2019/acs/acs5"
 
states_to_download = [
    ('01', 'AL'), ('02', 'AK'), ('04', 'AZ'), ('05', 'AR'), ('06', 'CA'),
    ('08', 'CO'), ('09', 'CT'), ('10', 'DE'), ('12', 'FL'), ('13', 'GA'),
    ('15', 'HI'), ('16', 'ID'), ('17', 'IL'), ('18', 'IN'), ('19', 'IA'),
    ('20', 'KS'), ('21', 'KY'), ('22', 'LA'), ('23', 'ME'), ('24', 'MD'),
    ('25', 'MA'), ('26', 'MI'), ('27', 'MN'), ('28', 'MS'), ('29', 'MO'),
    ('30', 'MT'), ('31', 'NE'), ('32', 'NV'), ('33', 'NH'), ('34', 'NJ'),
    ('35', 'NM'), ('36', 'NY'), ('37', 'NC'), ('38', 'ND'), ('39', 'OH'),
    ('40', 'OK'), ('41', 'OR'), ('42', 'PA'), ('44', 'RI'), ('45', 'SC'),
    ('46', 'SD'), ('47', 'TN'), ('48', 'TX'), ('49', 'UT'), ('50', 'VT'),
    ('51', 'VA'), ('53', 'WA'), ('54', 'WV'), ('55', 'WI'), ('56', 'WY'),
    ('11', 'DC'),  # DC
]


In [ ]:
acs_data = []

# downloading ACS data 
for state_code, state_name in states_to_download:
    print(f"{state_name}", end=" ", flush=True)
    
    #  build the request
    request_params = {
        'get': var_list,           # Which variables we want
        'for': 'tract:*',                # Get all census tracts
        'in': f'state:{state_code}',     # for the states
        'key': api_key                   #
    }
    
    try:
        response = requests.get(api_url, params=request_params, timeout=30)
        response.raise_for_status()  # Check for errors
        
        # parse the JSON response
        json_data = response.json()
        
        # convert to DataFrame (skip first row which is headers)
        df = pd.DataFrame(json_data[1:], columns=json_data[0])
        
        # save this state's data
        acs_data.append(df)
        
        print(f"{len(df)} tracts")
        
    except Exception as e:
        print(f"Error: {e}")
 
 
# combine all states into one dataframe
all_data = pd.concat(acs_data, ignore_index=True)
 
# create census tract ID
all_data['GEOID'] = all_data['state'] + all_data['county'] + all_data['tract']
 
# convert variable columns to numbers
for code in acs_variables.keys():
    if code in all_data.columns:
        all_data[code] = pd.to_numeric(all_data[code], errors='coerce')
 
# rename columns to readable names
all_data = all_data.rename(columns=acs_variables)
 

AL 1181 tracts
AK 167 tracts
AZ 1526 tracts
AR 686 tracts
CA 8057 tracts
CO 1249 tracts
CT 833 tracts
DE 218 tracts
FL 4245 tracts
GA 1969 tracts
HI 351 tracts
ID 298 tracts
IL 3123 tracts
IN 1511 tracts
IA 825 tracts
KS 770 tracts
KY 1115 tracts
LA 1148 tracts
ME 358 tracts
MD 1406 tracts
MA 1478 tracts
MI 2813 tracts
MN 1338 tracts
MS 664 tracts
MO 1393 tracts
MT 271 tracts
NE 532 tracts
NV 687 tracts
NH 295 tracts
NJ 2010 tracts
NM 499 tracts
NY 4918 tracts
NC 2195 tracts
ND 205 tracts
OH 2952 tracts
OK 1046 tracts
OR 834 tracts
PA 3218 tracts
RI 244 tracts
SC 1103 tracts
SD 222 tracts
TN 1497 tracts
TX 5265 tracts
UT 588 tracts
VT 184 tracts
VA 1907 tracts
WA 1458 tracts
WV 484 tracts
WI 1409 tracts
WY 132 tracts
DC 179 tracts


In [19]:
all_data.to_csv('acs_2015_2019.csv', index=False)

## Load CDC PLACES data
- Download directly from CDC website: https://chronicdata.cdc.gov/browse?category=Health+Outcomes

In [ ]:

places = pd.read_csv('PLACES__Census_Tract_Data_(GIS_Friendly_Format),_2021_release_20260503.csv') # cdc places dataset

### Load ACS data

In [ ]:
acs = pd.read_csv('acs_2015_2019.csv') #acs 5 year estimates

In [36]:
# Data Exploration

print(f"ACS: {len(acs)} rows, {acs.shape[1]} columns")
print(f"PLACES: {len(places)} rows, {places.shape[1]} columns")
 
# check column names in PLACES
print("\nPLACES columns:")
print(places.columns.tolist())

ACS: 73056 rows, 14 columns
PLACES: 72337 rows, 67 columns

PLACES columns:
['StateAbbr', 'StateDesc', 'CountyName', 'CountyFIPS', 'TractFIPS', 'TotalPopulation', 'ACCESS2_CrudePrev', 'ACCESS2_Crude95CI', 'ARTHRITIS_CrudePrev', 'ARTHRITIS_Crude95CI', 'BINGE_CrudePrev', 'BINGE_Crude95CI', 'BPHIGH_CrudePrev', 'BPHIGH_Crude95CI', 'BPMED_CrudePrev', 'BPMED_Crude95CI', 'CANCER_CrudePrev', 'CANCER_Crude95CI', 'CASTHMA_CrudePrev', 'CASTHMA_Crude95CI', 'CERVICAL_CrudePrev', 'CERVICAL_Crude95CI', 'CHD_CrudePrev', 'CHD_Crude95CI', 'CHECKUP_CrudePrev', 'CHECKUP_Crude95CI', 'CHOLSCREEN_CrudePrev', 'CHOLSCREEN_Crude95CI', 'COLON_SCREEN_CrudePrev', 'COLON_SCREEN_Crude95CI', 'COPD_CrudePrev', 'COPD_Crude95CI', 'COREM_CrudePrev', 'COREM_Crude95CI', 'COREW_CrudePrev', 'COREW_Crude95CI', 'CSMOKING_CrudePrev', 'CSMOKING_Crude95CI', 'DENTAL_CrudePrev', 'DENTAL_Crude95CI', 'DEPRESSION_CrudePrev', 'DEPRESSION_Crude95CI', 'DIABETES_CrudePrev', 'DIABETES_Crude95CI', 'GHLTH_CrudePrev', 'GHLTH_Crude95CI', 'HIGH

In [39]:
acs

,Population_Below_Poverty,Total_Population_Poverty,Median_Household_Income,Population_25Plus,Population_Less_9th_Grade,Civilian_Labor_Force,Unemployed,Employed,Total_Housing_Units,Vacant_Housing_Units,state,county,tract,GEOID
0,700,4655,37030,3285,270,3801,18,1955,2103,252,1,73,1100,1073001100
1,548,1946,36066,1274,222,1611,95,826,1148,332,1,73,1400,1073001400
2,2057,4007,27159,2802,654,3260,248,1803,1836,417,1,73,2000,1073002000
3,1040,5291,38721,3464,496,4115,245,2371,2477,583,1,73,3802,1073003802
4,1028,2533,18525,1971,336,2235,221,1045,1962,533,1,73,4000,1073004000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
73051,1371,3055,28639,1889,346,2200,253,1250,1303,182,11,1,7708,11001007708
73052,440,2298,41324,1558,197,1955,252,1174,949,119,11,1,7807,11001007807
73053,196,2318,90179,2062,71,2145,74,1584,1291,129,11,1,8402,11001008402
73054,864,4379,58621,3462,485,3752,157,2547,2015,204,11,1,8802,11001008802


In [40]:
places

,StateAbbr,StateDesc,CountyName,CountyFIPS,TractFIPS,TotalPopulation,ACCESS2_CrudePrev,ACCESS2_Crude95CI,ARTHRITIS_CrudePrev,ARTHRITIS_Crude95CI,...,OBESITY_Crude95CI,PHLTH_CrudePrev,PHLTH_Crude95CI,SLEEP_CrudePrev,SLEEP_Crude95CI,STROKE_CrudePrev,STROKE_Crude95CI,TEETHLOST_CrudePrev,TEETHLOST_Crude95CI,Geolocation
0,AZ,Arizona,Maricopa,4013,4013422643,5789,11.9,"(10.1, 14.0)",17.0,"(16.1, 17.9)",...,"(26.9, 29.7)",9.2,"( 8.2, 10.2)",34.5,"(33.0, 36.3)",1.8,"( 1.6, 2.0)",8.0,"( 5.2, 11.6)",POINT (-111.61853 33.35726769)
1,CA,California,Sacramento,6067,6067007402,6180,15.4,"(13.5, 17.3)",24.6,"(23.8, 25.3)",...,"(29.6, 31.4)",15.1,"(14.2, 16.2)",35.7,"(34.8, 36.7)",3.9,"( 3.6, 4.3)",18.2,"(13.7, 23.5)",POINT (-121.3791473 38.6869681)
2,AL,Alabama,Madison,1089,1089000201,760,25.4,"(21.2, 30.1)",36.0,"(34.6, 37.3)",...,"(46.6, 49.7)",22.5,"(20.4, 24.7)",50.3,"(48.8, 51.3)",7.6,"( 6.8, 8.6)",33.3,"(24.1, 43.8)",POINT (-86.55005486 34.77465775)
3,AL,Alabama,Montgomery,1101,1101002202,1185,25.2,"(21.2, 29.4)",36.1,"(35.0, 37.3)",...,"(45.3, 47.9)",20.7,"(19.1, 22.7)",49.7,"(48.3, 51.1)",7.2,"( 6.4, 8.0)",34.0,"(25.6, 42.0)",POINT (-86.30555503 32.31774882)
4,AL,Alabama,Butler,1013,1013952800,1394,14.3,"(11.9, 17.1)",36.0,"(34.5, 37.4)",...,"(32.3, 35.2)",15.2,"(13.5, 16.9)",35.7,"(33.9, 37.5)",4.2,"( 3.6, 4.8)",15.1,"( 8.9, 22.9)",POINT (-86.62833756 31.83774723)
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
72332,WI,Wisconsin,Milwaukee,55079,55079120101,3953,9.0,"( 7.2, 11.1)",30.4,"(28.9, 31.9)",...,"(28.4, 31.5)",10.3,"( 9.0, 11.8)",30.0,"(28.2, 31.3)",3.0,"( 2.5, 3.6)",8.6,"( 4.3, 15.3)",POINT (-88.04807056 42.95920626)
72333,WI,Wisconsin,Barron,55005,55005000300,4215,12.1,"(11.1, 13.2)",30.8,"(30.3, 31.4)",...,"(34.9, 36.4)",13.7,"(13.1, 14.3)",33.4,"(32.6, 34.3)",3.7,"( 3.5, 3.9)",14.4,"(12.2, 16.9)",POINT (-92.03012814 45.44804305)
72334,WI,Wisconsin,Ozaukee,55089,55089620100,6079,9.9,"( 8.5, 11.4)",25.3,"(24.3, 26.1)",...,"(33.7, 36.4)",10.8,"( 9.9, 11.7)",32.5,"(31.2, 33.6)",2.7,"( 2.5, 2.9)",10.8,"( 8.3, 13.6)",POINT (-87.98059035 43.41140163)
72335,WI,Wisconsin,Waushara,55137,55137960800,4515,15.8,"(13.8, 17.8)",31.9,"(30.9, 32.9)",...,"(38.3, 40.7)",15.5,"(14.3, 16.7)",34.0,"(33.1, 35.5)",4.4,"( 4.0, 4.8)",18.6,"(13.4, 23.7)",POINT (-89.31255284 44.06818511)


## Step 2 Data cleaning and merging 
### Merge keys:
  - PLACES: TractFIPS (census tract FIPS code)
  - ACS: GEOID (same format: state + county + tract)

### Merge type: Inner join
  - Keeps only tracts present in BOTH datasets

In [28]:
merged = pd.merge(
    places,
    acs,
    left_on='TractFIPS',
    right_on='GEOID',
    how='inner'
)

print(f"Merged: {len(merged)} rows")

# Select key columns
cols_to_keep = [
    'TractFIPS',
    'MHLTH_CrudePrev',      # Mental health
    'OBESITY_CrudePrev',    # Obesity
    'Median_Household_Income',
    'Poverty_Rate',
    'Unemployment_Rate',
    'Pct_Unemployed',
    'Pct_Less_9th_Grade',
    'Vacancy_Rate',
]


# Save
merged.to_csv('merged_data.csv', index=False)
print(merged.head())


Merged: 72335 rows
  StateAbbr   StateDesc  CountyName  CountyFIPS   TractFIPS  TotalPopulation  \
0        AZ     Arizona    Maricopa        4013  4013422643             5789   
1        CA  California  Sacramento        6067  6067007402             6180   
2        AL     Alabama     Madison        1089  1089000201              760   
3        AL     Alabama  Montgomery        1101  1101002202             1185   
4        AL     Alabama      Butler        1013  1013952800             1394   

   ACCESS2_CrudePrev ACCESS2_Crude95CI  ARTHRITIS_CrudePrev  \
0               11.9      (10.1, 14.0)                 17.0   
1               15.4      (13.5, 17.3)                 24.6   
2               25.4      (21.2, 30.1)                 36.0   
3               25.2      (21.2, 29.4)                 36.1   
4               14.3      (11.9, 17.1)                 36.0   

  ARTHRITIS_Crude95CI  ...  Population_Less_9th_Grade Civilian_Labor_Force  \
0        (16.1, 17.9)  ...                 

### DATA MERGE RESULTS:

**Merge key:** Census tract FIPS code (TractFIPS in PLACES = GEOID in ACS)
**Merge type:** Inner join (kept only matched tracts)

**Match summary:**
  - PLACES records: 72,337
  - ACS records: 73,056
  - Matched records: 72,335
  
  - Unmatched PLACES: 2 tracts (0.003%)
  - Unmatched ACS: 721 tracts (0.99%)
  
Decision: Used inner join to keep only tracts with both health outcome and socioeconomic data. The 2 unmatched PLACES tracts represent <0.01% of data and were dropped. The 721 unmatched ACS tracts likely represent smaller tracts not included in CDC PLACES sampling.

**Final analysis sample: 72,335 census tracts**